# LangChain Tools with Gemini API

이 노트북은 Google Gemini API를 사용하여 LangChain의 Tools 기능을 보여줍니다.

## 주요 학습 내용:
- 커스텀 도구(Tool) 생성 방법
- Gemini 모델에 도구 바인딩
- 자동 도구 호출 및 결과 처리
- 전 세계 시간대별 현재 시간 조회

## 시나리오:
사용자가 특정 지역의 현재 시간을 물어보면 AI가 자동으로 시간 조회 도구를 사용하여 답변

## 1. 환경 설정 및 모델 초기화

In [12]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from datetime import datetime
import pytz
import google.generativeai as genai

In [13]:
# 환경변수 로드
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("환경변수 GOOGLE_API_KEY가 설정되지 않았습니다.")
genai.configure(api_key=api_key)

print("✅ 환경변수가 성공적으로 로드되었습니다.")

✅ 환경변수가 성공적으로 로드되었습니다.


In [14]:
# Gemini 모델 초기화
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.7,
)

print("✅ Gemini 모델이 초기화되었습니다.")

✅ Gemini 모델이 초기화되었습니다.


## 2. 기본 채팅 테스트

In [15]:
# 기본 채팅 테스트
response = llm.invoke([HumanMessage("잘 지냈어?")])
print(f"🤖 Gemini 응답: {response.content}")

🤖 Gemini 응답: 네, 잘 지냈어요! 덕분에 오늘 하루도 활기차게 시작했답니다. 😊 혹시 오늘 특별한 일정이 있으신가요? 아니면 그냥 편안한 하루를 보내실 예정이신가요?


## 3. 커스텀 도구(Tool) 정의

In [16]:
@tool  # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    try:
        tz = pytz.timezone(timezone)
        now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        location_and_local_time = f'{timezone} ({location}) 현재시각 {now}'
        print(f"🕐 시간 조회: {location_and_local_time}")
        return location_and_local_time
    except Exception as e:
        error_msg = f"시간 조회 실패: {str(e)}"
        print(f"❌ {error_msg}")
        return error_msg

print("✅ 시간 조회 도구가 정의되었습니다.")
print(f"📋 도구 이름: {get_current_time.name}")
print(f"📝 도구 설명: {get_current_time.description}")

✅ 시간 조회 도구가 정의되었습니다.
📋 도구 이름: get_current_time
📝 도구 설명: 현재 시각을 반환하는 함수

Args:
    timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
    location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨


## 4. 도구를 모델에 바인딩

In [17]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time]
tool_dict = {"get_current_time": get_current_time}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

print("✅ 도구가 모델에 바인딩되었습니다.")
print(f"📋 사용 가능한 도구: {[tool.name for tool in tools]}")

✅ 도구가 모델에 바인딩되었습니다.
📋 사용 가능한 도구: ['get_current_time']


## 5. 첫 번째 도구 호출 테스트 - 부산 시간 조회

In [18]:
# 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

print("💬 사용자 질문: 부산은 지금 몇시야?")

# llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

print(f"\n🤖 AI 응답: {response.content}")
print(f"\n📋 메시지 히스토리 길이: {len(messages)}")
print(f"📋 응답 타입: {type(response)}")

💬 사용자 질문: 부산은 지금 몇시야?

🤖 AI 응답: 

📋 메시지 히스토리 길이: 3
📋 응답 타입: <class 'langchain_core.messages.ai.AIMessage'>


In [19]:
# tool_calls 확인 및 처리
if hasattr(response, 'tool_calls') and response.tool_calls:
    print(f"🔧 호출된 도구 수: {len(response.tool_calls)}")
    
    for tool_call in response.tool_calls:
        selected_tool = tool_dict[tool_call["name"]]
        print(f"🛠️ 도구 호출: {tool_call['name']}")
        print(f"📥 전달된 인자: {tool_call['args']}")
        
        # 도구 함수를 호출하여 결과를 반환
        tool_msg = selected_tool.invoke(tool_call)
        messages.append(tool_msg)
        print(f"📤 도구 실행 결과: {tool_msg.content}")
else:
    print("ℹ️ 도구 호출이 없었습니다.")

print(f"\n📋 현재 메시지 수: {len(messages)}")

🔧 호출된 도구 수: 1
🛠️ 도구 호출: get_current_time
📥 전달된 인자: {'timezone': 'Asia/Seoul', 'location': '부산'}
🕐 시간 조회: Asia/Seoul (부산) 현재시각 2025-09-04 10:26:29
📤 도구 실행 결과: Asia/Seoul (부산) 현재시각 2025-09-04 10:26:29

📋 현재 메시지 수: 4


In [20]:
# 도구 실행 결과를 바탕으로 최종 답변 생성
if len(messages) > 2:  # 도구 실행 결과가 있다면
    print("🔄 도구 실행 결과를 바탕으로 최종 답변 생성 중...")
    final_response = llm_with_tools.invoke(messages)
    print(f"\n🎯 최종 답변: {final_response.content}")
else:
    print("💬 도구 실행 없이 직접 답변했습니다.")

🔄 도구 실행 결과를 바탕으로 최종 답변 생성 중...

🎯 최종 답변: 부산은 현재 2025년 9월 4일 10시 26분 29초입니다.


## 6. 전체 메시지 히스토리 확인

In [21]:
# 전체 메시지 히스토리 출력
print("📜 전체 메시지 히스토리:")
print("=" * 50)

for i, message in enumerate(messages, 1):
    message_type = message.__class__.__name__
    if hasattr(message, 'content'):
        content = message.content[:100] + "..." if len(message.content) > 100 else message.content
        print(f"{i}. {message_type}: {content}")
    else:
        print(f"{i}. {message_type}: [메타데이터만 포함]")
    print("-" * 30)

📜 전체 메시지 히스토리:
1. SystemMessage: 너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.
------------------------------
2. HumanMessage: 부산은 지금 몇시야?
------------------------------
3. AIMessage: 
------------------------------
4. ToolMessage: Asia/Seoul (부산) 현재시각 2025-09-04 10:26:29
------------------------------


## 7. 다양한 지역 시간 조회 테스트

In [22]:
# 여러 지역 시간 조회 테스트
test_questions = [
    "서울은 지금 몇시야?",
    "도쿄의 현재 시간은?",
    "뉴욕은 지금 몇시지?",
    "런던의 현재 시간을 알려줘"
]

print("🌍 여러 지역 시간 조회 테스트")
print("=" * 40)

for question in test_questions:
    print(f"\n💬 질문: {question}")
    
    messages = [
        SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
        HumanMessage(question),
    ]
    
    try:
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        if hasattr(response, 'tool_calls') and response.tool_calls:
            for tool_call in response.tool_calls:
                selected_tool = tool_dict[tool_call["name"]]
                tool_msg = selected_tool.invoke(tool_call)
                messages.append(tool_msg)
            
            final_response = llm_with_tools.invoke(messages)
            print(f"🎯 답변: {final_response.content}")
        else:
            print(f"🤖 답변: {response.content}")
            
    except Exception as e:
        print(f"❌ 오류 발생: {str(e)}")

🌍 여러 지역 시간 조회 테스트

💬 질문: 서울은 지금 몇시야?
🕐 시간 조회: Asia/Seoul (서울) 현재시각 2025-09-04 10:26:30
🎯 답변: 서울은 현재 2025년 9월 4일 10시 26분 30초입니다.

💬 질문: 도쿄의 현재 시간은?
🕐 시간 조회: Asia/Tokyo (도쿄) 현재시각 2025-09-04 10:26:32
🎯 답변: 도쿄의 현재 시각은 2025년 9월 4일 10시 26분 32초입니다.

💬 질문: 뉴욕은 지금 몇시지?
🕐 시간 조회: America/New_York (New York) 현재시각 2025-09-03 21:26:34
🎯 답변: 뉴욕은 지금 2025년 9월 3일 21시 26분 34초입니다.

💬 질문: 런던의 현재 시간을 알려줘
🕐 시간 조회: Europe/London (런던) 현재시각 2025-09-04 02:26:38
🎯 답변: 런던의 현재 시각은 2025년 9월 4일 02시 26분 38초입니다.


## 8. 일반 질문 테스트 (도구 사용 불필요)

In [23]:
# 도구가 필요하지 않은 일반 질문들
general_questions = [
    "안녕하세요! 어떻게 지내세요?",
    "Python에서 리스트와 튜플의 차이점은?",
    "오늘 날씨가 좋네요",
    "LangChain이 뭐야?"
]

print("💬 일반 질문 테스트 (도구 사용 불필요)")
print("=" * 40)

for question in general_questions:
    print(f"\n💬 질문: {question}")
    
    messages = [
        SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다. 하지만 도구가 필요하지 않은 일반적인 질문에는 직접 답변해도 된다."),
        HumanMessage(question),
    ]
    
    try:
        response = llm_with_tools.invoke(messages)
        print(f"🤖 답변: {response.content}")
        
        if hasattr(response, 'tool_calls') and response.tool_calls:
            print("   🔧 도구 호출됨")
        else:
            print("   💬 일반 답변")
            
    except Exception as e:
        print(f"❌ 오류 발생: {str(e)}")

💬 일반 질문 테스트 (도구 사용 불필요)

💬 질문: 안녕하세요! 어떻게 지내세요?
🤖 답변: 저는 잘 지내고 있습니다. 무엇을 도와드릴까요?
   💬 일반 답변

💬 질문: Python에서 리스트와 튜플의 차이점은?
🤖 답변: 파이썬에서 리스트와 튜플은 모두 여러 항목을 저장하는 데 사용되는 데이터 구조이지만 몇 가지 중요한 차이점이 있습니다.

* **변경 가능성:** 리스트는 변경 가능하지만 튜플은 변경 불가능합니다. 즉, 리스트를 만든 후에는 항목을 추가, 제거 또는 수정할 수 있지만 튜플을 만든 후에는 변경할 수 없습니다.
* **구문:** 리스트는 대괄호(`[]`)를 사용하여 정의하는 반면 튜플은 괄호(`()`)를 사용하여 정의합니다.
* **성능:** 튜플은 변경 불가능하기 때문에 리스트보다 빠를 수 있습니다.
* **용도:** 리스트는 항목을 변경해야 할 때 사용하기에 좋고, 튜플은 항목을 변경하지 않아야 할 때 사용하기에 좋습니다.

다음은 몇 가지 예입니다.

```python
# 리스트
my_list = [1, 2, 3]
my_list[0] = 4  # 리스트의 첫 번째 항목을 변경합니다.
my_list.append(5)  # 리스트에 새 항목을 추가합니다.

# 튜플
my_tuple = (1, 2, 3)
my_tuple[0] = 4  # 오류가 발생합니다. 튜플은 변경 불가능하기 때문입니다.
```
   💬 일반 답변

💬 질문: 오늘 날씨가 좋네요
🤖 답변: 네, 좋은 하루 보내세요! 혹시 현재 시간을 알고 싶으시면 알려드릴 수 있습니다.
   💬 일반 답변

💬 질문: LangChain이 뭐야?
🤖 답변: 저는 LangChain에 대해 잘 알지 못합니다. 저는 현재 시각을 알려드릴 수 있습니다.
   💬 일반 답변


## 9. 사용 가능한 타임존 정보

In [24]:
# 사용 가능한 타임존 예시
sample_timezones = [
    ("Asia/Seoul", "서울"),
    ("Asia/Tokyo", "도쿄"),
    ("America/New_York", "뉴욕"),
    ("Europe/London", "런던"),
    ("America/Los_Angeles", "로스앤젤레스"),
    ("Australia/Sydney", "시드니"),
    ("Europe/Paris", "파리"),
    ("Asia/Shanghai", "상하이"),
]

print("🌍 주요 도시별 타임존 정보")
print("=" * 40)
print("\n📍 지원하는 주요 도시들:")

for timezone, city in sample_timezones:
    print(f"  🏙️ {city}: {timezone}")

print("\n💡 사용법:")
print("  - '서울은 지금 몇시야?' 형태로 질문")
print("  - AI가 자동으로 적절한 타임존을 선택하여 시간 조회")
print("  - 전 세계 모든 pytz 타임존 지원")

🌍 주요 도시별 타임존 정보

📍 지원하는 주요 도시들:
  🏙️ 서울: Asia/Seoul
  🏙️ 도쿄: Asia/Tokyo
  🏙️ 뉴욕: America/New_York
  🏙️ 런던: Europe/London
  🏙️ 로스앤젤레스: America/Los_Angeles
  🏙️ 시드니: Australia/Sydney
  🏙️ 파리: Europe/Paris
  🏙️ 상하이: Asia/Shanghai

💡 사용법:
  - '서울은 지금 몇시야?' 형태로 질문
  - AI가 자동으로 적절한 타임존을 선택하여 시간 조회
  - 전 세계 모든 pytz 타임존 지원


## 10. 직접 도구 테스트

In [25]:
# 도구를 직접 호출해보기
print("🔧 도구 직접 호출 테스트")
print("=" * 30)

# 직접 도구 호출
test_locations = [
    ("Asia/Seoul", "서울"),
    ("America/New_York", "뉴욕"),
    ("Europe/London", "런던"),
    ("Invalid/Timezone", "잘못된 지역")
]

for timezone, location in test_locations:
    print(f"\n🌍 {location} ({timezone}) 시간 조회:")
    result = get_current_time.invoke({"timezone": timezone, "location": location})
    print(f"📅 결과: {result}")

🔧 도구 직접 호출 테스트

🌍 서울 (Asia/Seoul) 시간 조회:
🕐 시간 조회: Asia/Seoul (서울) 현재시각 2025-09-04 10:26:44
📅 결과: Asia/Seoul (서울) 현재시각 2025-09-04 10:26:44

🌍 뉴욕 (America/New_York) 시간 조회:
🕐 시간 조회: America/New_York (뉴욕) 현재시각 2025-09-03 21:26:44
📅 결과: America/New_York (뉴욕) 현재시각 2025-09-03 21:26:44

🌍 런던 (Europe/London) 시간 조회:
🕐 시간 조회: Europe/London (런던) 현재시각 2025-09-04 02:26:44
📅 결과: Europe/London (런던) 현재시각 2025-09-04 02:26:44

🌍 잘못된 지역 (Invalid/Timezone) 시간 조회:
❌ 시간 조회 실패: 'Invalid/Timezone'
📅 결과: 시간 조회 실패: 'Invalid/Timezone'


## 11. 요약 및 학습 정리

In [26]:
print("🎉 LangChain Tools with Gemini API 완료!")
print("=" * 50)

print("\n📚 학습한 주요 개념:")
print("  1️⃣ @tool 데코레이터를 사용한 커스텀 도구 생성")
print("  2️⃣ bind_tools()로 모델에 도구 바인딩")
print("  3️⃣ 자동 도구 호출 및 tool_calls 처리")
print("  4️⃣ 도구 실행 결과를 활용한 자연스러운 답변 생성")
print("  5️⃣ 일반 질문과 도구 필요 질문의 자동 구분")

print("\n🔧 구현한 주요 기능:")
print("  - 전 세계 시간대별 현재 시간 조회")
print("  - 에러 처리 및 복구")
print("  - 자동 도구 선택 및 실행")
print("  - 메시지 히스토리 관리")

print("\n🛠️ 사용한 주요 도구:")
print("  - Google Gemini 2.0 Flash 모델")
print("  - LangChain Tools 프레임워크")
print("  - pytz 타임존 라이브러리")
print("  - datetime 시간 처리")

print("\n💡 핵심 포인트:")
print("  ✅ AI가 필요에 따라 자동으로 도구를 선택하고 실행")
print("  ✅ 도구 실행 결과를 바탕으로 자연스러운 답변 생성")
print("  ✅ 에러 처리와 복구 메커니즘 구현")
print("  ✅ 도구가 필요 없는 질문은 직접 답변")

print("\n🚀 확장 가능한 방향:")
print("  - 날씨 조회, 뉴스 검색, 계산기 등 다양한 도구 추가")
print("  - 여러 도구를 조합한 복합 작업 처리")
print("  - 도구 실행 결과 캐싱 및 최적화")
print("  - 사용자별 개인화된 도구 설정")

🎉 LangChain Tools with Gemini API 완료!

📚 학습한 주요 개념:
  1️⃣ @tool 데코레이터를 사용한 커스텀 도구 생성
  2️⃣ bind_tools()로 모델에 도구 바인딩
  3️⃣ 자동 도구 호출 및 tool_calls 처리
  4️⃣ 도구 실행 결과를 활용한 자연스러운 답변 생성
  5️⃣ 일반 질문과 도구 필요 질문의 자동 구분

🔧 구현한 주요 기능:
  - 전 세계 시간대별 현재 시간 조회
  - 에러 처리 및 복구
  - 자동 도구 선택 및 실행
  - 메시지 히스토리 관리

🛠️ 사용한 주요 도구:
  - Google Gemini 2.0 Flash 모델
  - LangChain Tools 프레임워크
  - pytz 타임존 라이브러리
  - datetime 시간 처리

💡 핵심 포인트:
  ✅ AI가 필요에 따라 자동으로 도구를 선택하고 실행
  ✅ 도구 실행 결과를 바탕으로 자연스러운 답변 생성
  ✅ 에러 처리와 복구 메커니즘 구현
  ✅ 도구가 필요 없는 질문은 직접 답변

🚀 확장 가능한 방향:
  - 날씨 조회, 뉴스 검색, 계산기 등 다양한 도구 추가
  - 여러 도구를 조합한 복합 작업 처리
  - 도구 실행 결과 캐싱 및 최적화
  - 사용자별 개인화된 도구 설정
